> 在线讲义：[第 1 章 · 交叉熵 loss：模型猜错了多少，怎么量](https://wikieden.github.io/beyond-transformer/lessons/01-cross-entropy.html) · 本书《超越 Transformer》

# 第 1 章 · 交叉熵 loss：模型猜错了多少，怎么量

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/01-cross-entropy.html) · 仓库里 `beyond-transformer/notes/lessons/01-cross-entropy.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第三步 −log p | $\ell = -\log p$ | `nll(p)` | 0.9 → 0.105，0.537 → 0.622，0.05 → 2.996 |
| 第三步 乘法变加法 | $\log(ab) = \log a + \log b$ | `np.log(0.1 * 0.01)` | 两边都是 −6.908 |
| 第四步 式 16 | 取平均 | `dataset_loss([0.5, 0.8, 0.4, 0.9])` | 0.484 |
| 第四步 式 17 | −o[正确] + log Σ e^o | `ce_from_logits(LOGITS, 6)` | 0.622（和模型 A 一样） |
| 第四步 减最大值 | | `ce_from_logits([1000, 999, 998], 0)` | 0.408，不溢出 |
| 第五步 困惑度 | exp(平均 loss) | `perplexity(...)` | 两句话 1.62；4 个词瞎猜 4 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
LOGITS = np.array([0.7, -1.7, 1.7, -1.6, 2.0, -1.5, 2.9, -1.6])  # 前作第 22 章那 8 个分数，bites 是第 7 个（下标 6）

**函数 `nll`**：一次猜测的 loss：−log(正确词的概率)。

In [ ]:
def nll(p):
    """一次猜测的 loss：−log(正确词的概率)。"""
    return -np.log(p)

**函数 `dataset_loss`**：式 16：所有句子、所有位置的 −log p 取平均。

In [ ]:
def dataset_loss(probs):
    """式 16：所有句子、所有位置的 −log p 取平均。"""
    return float(np.mean([nll(p) for p in probs]))

**函数 `ce_from_logits`**：式 17 化简：−o[target] + log Σ e^o，先减最大值防溢出。

In [ ]:
def ce_from_logits(o, target):
    """式 17 化简：−o[target] + log Σ e^o，先减最大值防溢出。"""
    o = np.asarray(o, float)
    m = o.max()
    return float(-o[target] + m + np.log(np.exp(o - m).sum()))

**函数 `perplexity`**：式 18：exp(平均 loss)。

In [ ]:
def perplexity(losses):
    """式 18：exp(平均 loss)。"""
    return float(np.exp(np.mean(losses)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步 −log p：模型 B、A、C")
for name, p in (("B", 0.9), ("A", 0.537), ("C", 0.05)):
    print(f"模型 {name}：p = {p}，扣分 {nll(p):.3f}，对照 1 − p = {1 - p:.3f}")
print("log(0.1 × 0.01) =", round(np.log(0.1 * 0.01), 3), " log 0.1 + log 0.01 =", round(np.log(0.1) + np.log(0.01), 3))

In [ ]:
print("\n第四步 式 16：两句话 4 次猜测")
probs = [0.5, 0.8, 0.4, 0.9]
print("平均 loss =", round(dataset_loss(probs), 3))
print("式 17：bites 的 loss =", round(ce_from_logits(LOGITS, 6), 3))
print("[1000, 999, 998]，正确词是第一个：", round(ce_from_logits([1000, 999, 998], 0), 3))

In [ ]:
print("\n第五步 困惑度")
print("两句话：", round(perplexity([nll(p) for p in probs]), 2))
print("4 个词里瞎猜：", round(perplexity([np.log(4)]), 2))

## 3. 代码里才看得到的

- `np.log` 就是以 e 为底的 ln；以 10 为底要写 `np.log10`。
- bites 在讲义里是"第 7 个"，代码里下标从 0 数，是 `LOGITS[6]`。

## 4. 自己改一改

- 直接写 `np.log(np.exp(np.array([1000.0])).sum())`，看不减最大值时得到什么（inf）。
- 把 `dataset_loss` 里某个 0.9 改成 0.99，看平均 loss 和困惑度怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 句子自己当答案，只看正确那一格。
> log 问"e 的几次方"，1 是零、越小越负。
> 扣分就取负 log：猜对不扣，越错越狠没上限；连乘变相加，整句不怕小。
> 全部位置取平均，一次前向全拿到；分数直接算，减去最大不溢出。
> 平均 loss 取 exp，就是困惑度：相当于在几个词里犹豫。